# 01. Waste Dataset Exploration & Quality Audit
**wasteinfoAI — Phase 3: Waste Classification Model**

This notebook provides exploratory data analysis (EDA), stratified dataset splitting, class imbalance analysis, and visual image auditing for the **17 waste categories** in `balanced_waste_images`:
- `Battery`, `cardboard`, `glass`, `Keyboard`, `metal`, `Microwave`, `Mobile`, `Mouse`
- `organic`, `paper`, `PCB`, `plastic`, `Player`, `Printer`, `Television`, `trash`, `Washing Machine`

### Pipeline Structure
1. **Custom Datasets & DataLoaders**: Stratified train/val/test splitting.
2. **Data Analysis & Visualization**: Class imbalance bar charts, pie charts, and sample grid inspection.

## 1. Custom Datasets & DataLoaders
Load images with stratified train (80%), validation (10%), and test (10%) splits.

In [ ]:
import os
import sys
from pathlib import Path
import torch
from torchvision import transforms as T

# Add src to module search path
ML_ROOT = Path("..").resolve()
sys.path.insert(0, str(ML_ROOT))

from src.data.custom_dataset import CustomDataset, resolve_dataset_root
from src.data.visualization import Visualization

# Resolve dataset root (supports Kaggle / Colab / local workspace)
dataset_root = resolve_dataset_root()
print(f"Dataset Root: {dataset_root}")

# Preprocessing transforms
mean, std, im_size, bs = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 224, 32
tfs = T.Compose([
    T.Resize((im_size, im_size)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])

# Create stratified DataLoaders
tr_dl, val_dl, ts_dl, classes, cls_counts = CustomDataset.stratified_split_dls(
    root=dataset_root,
    transformations=tfs,
    bs=bs,
    ns=0 if sys.platform == 'win32' else 2,
    seed=2025
)

print(f"Train batches: {len(tr_dl)}")
print(f"Val batches:   {len(val_dl)}")
print(f"Test batches:  {len(ts_dl)}")
print(f"Total classes: {len(classes)}")

## 2. Data Analysis
Analyze class sample distributions, inspect imbalance across train/val/test splits, and preview normalized image tensors.

In [ ]:
# Initialize Visualization instance
vis = Visualization(
    vis_datas=[tr_dl, val_dl, ts_dl],
    n_ims=20,
    rows=4,
    cmap="rgb",
    cls_names=list(classes.keys()),
    cls_counts=cls_counts
)

# 1. Plot class imbalance bar chart across train/val/test splits
vis.analysis()

### Class Distribution Pie Charts

In [ ]:
# 2. Plot class distribution percentage pie chart
vis.pie_chart()

### Sample Image Visualization Grid
Inspect randomly sampled images with ground-truth (GT) labels inverted from ImageNet normalization.

In [ ]:
# 3. Grid visualization of batch images with ground truth labels
vis.visualization()

## 3. Phase 3 Audit Summary
- **Class Coverage:** 17 classes with ~400 balanced images each.
- **Stratified Splitting:** 80% train, 10% validation, 10% held-out test.
- **Parity Verified:** Preprocessing matches `classifier_config.yaml` and Android CameraX normalization contract.